# Python Tutorial for Beginner
初心者を対象に、pythonを用いて、機械学習の一連の流れを理解することを目的とする。

In [ ]:
! pip install pandas matplotlib seaborn
#Google Colab用
! wget https://raw.githubusercontent.com/ChemicalBatteryLab-Nitech/dxgem_day4day5/main/Day4_Part1/olivineDataset_withEA.csv
! wget https://raw.githubusercontent.com/ChemicalBatteryLab-Nitech/dxgem_day4day5/main/Day4_Part1/validation.csv


### 棒グラフを作る


### 📝 このコードの想定プロンプト（①シンプル棒グラフ）

> `olivineDataset_withEA.csv` を読み込み、'EA (eV)' 列に欠損値がある行を除いたうえで、
> 'Composition' 列をx軸、'EA (eV)' 列をy軸にした棒グラフを matplotlib で描いてください。
> x軸のラベルは多いので縦書き（rotation=90）にしてください。

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# --- CSVファイルの読み込み ---
df = pd.read_csv("olivineDataset_withEA.csv")

# --- 欠損データの除去 ---
df_valid = df.dropna(subset=['EA (eV)'])

# --- 棒グラフの作成 ---
plt.figure(figsize=(12, 6))
plt.bar(df_valid['Composition'], df_valid['EA (eV)'], color='skyblue')

# --- 軸ラベル・タイトルなど ---
plt.xlabel('Composition')
plt.ylabel('EA (eV)')
plt.title('Electron Affinity (EA) of Olivine Compositions')
plt.xticks(rotation=90, fontsize=8)  # ラベルが多い場合の回転とサイズ調整
plt.tight_layout()
plt.show()


### 📝 このコードの想定プロンプト（②カラーマップ付き棒グラフ）

> 上の棒グラフをベースに、各棒の色を EA 値の大小に対応した
> カラーマップ（viridis）で塗り分けてください。
> カラーバーも右側に表示してください。
> seaborn の whitegrid スタイルも適用してください。

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# --- seabornスタイル ---
sns.set(style="whitegrid")

# --- データ読み込みと前処理 ---
df = pd.read_csv("olivineDataset_withEA.csv")
df_valid = df.dropna(subset=['EA (eV)'])  # ← カラム名に注意

# --- 色の正規化とカラーマップの準備 ---
ea_values = df_valid['EA (eV)'].values
norm = plt.Normalize(ea_values.min(), ea_values.max())
cmap = plt.cm.viridis
colors = cmap(norm(ea_values))

# --- フィギュアと軸の定義 ---
fig, ax = plt.subplots(figsize=(14, 6))
bars = ax.bar(df_valid['Composition'], ea_values, color=colors, edgecolor='black', linewidth=0.5)

# --- 軸とタイトル ---
ax.set_xlabel('Composition', fontsize=12, weight='bold')
ax.set_ylabel('EA (eV)', fontsize=12, weight='bold')
ax.set_title('Electron Affinity of Olivine Compositions', fontsize=14, weight='bold')
ax.tick_params(axis='x', rotation=90, labelsize=9)
ax.tick_params(axis='y', labelsize=10)
ax.grid(axis='y', linestyle='--', alpha=0.5)

# --- カラーバー ---
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array(ea_values)
cbar = fig.colorbar(sm, ax=ax, aspect=30, pad=0.02)
cbar.set_label('EA (eV)', fontsize=11)

# --- レイアウト調整 ---
plt.tight_layout()
plt.show()


### 記述子の相関係数行列の可視化

### 📝 このコードの想定プロンプト（③相関係数ヒートマップ）

> `olivineDataset_withEA.csv` を読み込み、先頭2列（Composition, EA）を除いた
> 記述子列全体について相関係数行列を計算し、seaborn の heatmap で可視化してください。
> カラーマップは coolwarm を使い、欠損値がある行は除いてください。

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# --- データ読み込み（1行目をヘッダーとして利用）---
df = pd.read_csv("olivineDataset_withEA.csv", header=0)

# --- C列〜AS列の抽出（0-indexedで2〜44列目）---
df_descriptors = df.iloc[:, 2:45]

# --- 空欄のある行を削除 ---
df_clean = df_descriptors.dropna()

# --- 相関係数の計算 ---
corr_matrix = df_clean.corr()

# --- ヒートマップの描画 ---
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, cmap='coolwarm', annot=False, square=True, 
            cbar_kws={"shrink": 0.8}, xticklabels=True, yticklabels=True)

plt.title("Correlation Heatmap of Descriptors (C to AS)")
plt.tight_layout()
plt.show()


### 診断プロットをつくる

### 📝 このコードの想定プロンプト（④カーネル密度付き診断プロット）

> `validation.csv`（ヘッダなし2列：1列目=予測値、2列目=実測値）を読み込み、
> 点の密度をカラーで示した parity plot（散布図）を描いてください。
> あわせて y = x + b のオフセット付き直線フィット、RMSE、R² をプロット上に表示してください。
> 点の密度推定には scipy の gaussian_kde を使ってください。

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, r2_score
from scipy.stats import gaussian_kde

# --- ファイル読み込み ---
df = pd.read_csv("validation.csv", header=None)
x = df.iloc[:, 0].values
y = df.iloc[:, 1].values



# --- y = x + b にフィッティング（傾き1を仮定し切片bを最小二乗で求める）---
b = np.mean(y - x)

# --- 密度推定 ---
xy = np.vstack([x, y])
z = gaussian_kde(xy)(xy)
idx = z.argsort()
x, y, z = x[idx], y[idx], z[idx]

# --- プロット ---
plt.figure(figsize=(6, 6))
sc = plt.scatter(x, y, c=z, cmap='jet', s=10, edgecolor='none')

# y = x + b の直線を描画
x_line = np.linspace(min(x.min(), y.min()), max(x.max()+2, y.max()+2), 100)
plt.plot(x_line, x_line + b, 'k--', linewidth=1, label=f'y = x + {b:.3f}')

# --- RMSEとR2の計算 ---
rmse = np.sqrt(mean_squared_error(y, x+b))
r2 = r2_score(y, x+b)


# --- 表示設定 ---
plt.xlabel("Predicted [eV]")
plt.ylabel("True [eV]")
plt.xlim([x.min()-1,x.max()+1])
plt.ylim([y.min()-1,y.max()+1])
plt.title("Validation: Predicted vs True")
plt.colorbar(sc, label='Density')
plt.legend()

# --- テキスト表示 ---
plt.text(0.05, 0.95, f"RMSE = {rmse:.3f} eV\n$R^2$ = {r2:.3f}\nOffset b = {b:.3f} eV",
         transform=plt.gca().transAxes,
         fontsize=10, verticalalignment='top',
         bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))

plt.tight_layout()
plt.show()


